# Dog Stool Detection Model - Example Usage

This notebook demonstrates how to use the dog stool detection model for training and inference.

## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import yaml
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from src.model import create_model, MultiTaskStoolClassifier
from src.dataset import DogStoolDataset, get_train_transforms, get_val_transforms
from src.inference import StoolDetector
from src.utils import count_parameters

%matplotlib inline

## 2. Load Configuration

In [ ]:
# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"Model backbone: {config['model']['backbone']}")
print(f"Image size: {config['data']['image_size']}")
print(f"Batch size: {config['data']['batch_size']}")

## 3. Create and Inspect Model

In [ ]:
# Create model
model = create_model(config)

# Count parameters
total_params, trainable_params = count_parameters(model)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
dummy_input = torch.randn(1, 3, 224, 224)
outputs = model(dummy_input)

print("\nModel outputs:")
for task_name, output in outputs.items():
    print(f"{task_name}: {output.shape}")

## 4. Dataset Inspection

In [ ]:
# Create dataset
label_config = config['labels']
image_size = tuple(config['data']['image_size'])

train_dataset = DogStoolDataset(
    data_dir='../data/train',
    label_config=label_config,
    transform=get_val_transforms(image_size),  # Use val transforms for visualization
    image_size=image_size
)

print(f"Dataset size: {len(train_dataset)}")
print(f"\nLabel categories:")
for task_name, classes in label_config.items():
    print(f"{task_name}: {len(classes)} classes")
    print(f"  - {', '.join(classes)}")

## 5. Visualize Sample Data

In [ ]:
# Visualize first sample if dataset is not empty
if len(train_dataset) > 0:
    image, labels = train_dataset[0]
    
    # Denormalize image for visualization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image_vis = image * std + mean
    image_vis = torch.clamp(image_vis, 0, 1)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(image_vis.permute(1, 2, 0))
    plt.axis('off')
    plt.title('Sample Image')
    plt.show()
    
    print("Labels:")
    for task_name, label_idx in labels.items():
        class_name = label_config[task_name][label_idx.item()]
        print(f"{task_name}: {class_name}")
else:
    print("Dataset is empty. Please add images and labels first.")

## 6. Training (Optional)

You can train the model directly in the notebook or use the training script.

In [ ]:
# Uncomment to train in notebook
# from src.train import Trainer

# trainer = Trainer(config, device='cuda')
# trainer.train()

## 7. Inference with Trained Model

In [ ]:
# Load trained model for inference
checkpoint_path = '../checkpoints/best_model.pth'

if Path(checkpoint_path).exists():
    detector = StoolDetector(
        config_path='../config.yaml',
        checkpoint_path=checkpoint_path
    )
    
    # Example prediction
    # results = detector.analyze_image('path/to/image.jpg')
    print("Model loaded successfully!")
else:
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Please train the model first.")

## 8. Batch Prediction Example

In [ ]:
# Batch prediction on test set
if Path(checkpoint_path).exists():
    test_image_dir = '../data/test/images'
    
    if Path(test_image_dir).exists():
        results = detector.batch_predict(
            image_dir=test_image_dir,
            output_file='../test_results.json'
        )
        
        print(f"\nProcessed {len(results)} images")
    else:
        print(f"Test directory not found: {test_image_dir}")

## 9. Model Performance Analysis

In [ ]:
# Analyze prediction confidence distribution
if Path(checkpoint_path).exists() and 'results' in locals():
    import numpy as np
    
    # Collect confidences for each task
    confidences = {task: [] for task in config['labels'].keys()}
    
    for result in results:
        for task_name, pred in result['predictions'].items():
            confidences[task_name].append(pred['confidence'])
    
    # Plot confidence distributions
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, (task_name, conf_list) in enumerate(confidences.items()):
        if conf_list:
            axes[idx].hist(conf_list, bins=20, edgecolor='black')
            axes[idx].set_title(f'{task_name} Confidence Distribution')
            axes[idx].set_xlabel('Confidence')
            axes[idx].set_ylabel('Count')
            axes[idx].axvline(np.mean(conf_list), color='red', linestyle='--', 
                             label=f'Mean: {np.mean(conf_list):.3f}')
            axes[idx].legend()
    
    plt.tight_layout()
    plt.show()

## Conclusion

This notebook demonstrated:
1. Loading model configuration
2. Creating and inspecting the model
3. Working with datasets
4. Running inference
5. Analyzing results

For production use, refer to the training and inference scripts in the `src/` directory.